# `ar-db-handler` — GCS path resolution playground (v0.4.0)

A focused tour through the new `paths.py` module and the auto-fill
behaviour added to `upsert_file()` in v0.4.0.

> **This notebook never touches a real database or a real GCS bucket.**
> Every cell writes to a fresh `tempfile.TemporaryDirectory()` that
> lives only for the lifetime of this kernel. The path strings we
> produce would point at GCS in production — here they're just
> strings.

We'll walk through:

1. Initialising `filings.db` + seeding companies and a scraper run.
2. The canonical path scheme via **`make_blob_path()`**.
3. How each malformed component triggers a precise, field-naming error.
4. **`resolve_gcs_path()`** — the `FileRecord` wrapper.
5. **`derive_fiscal_year()`** — the H2/H1 rule, the single source of
   truth for the scrapers.
6. **The headline behaviour**: `upsert_file()` auto-fills
   `country_code` (from the `companies` table) and `gcs_path`
   (via `resolve_gcs_path`) on SUCCESS rows when the caller omits them.
7. Caller-wins semantics — explicit `gcs_path` is written verbatim.
8. PENDING-row exemption.
9. The two new error paths: `ERROR_FK_VIOLATION` on unknown company,
   `ERROR_MISSING_REPORTING_DATE` when the date can't be resolved.
10. A typical scraper flow tying everything together.

## 0. Setup — a throwaway working directory

Standard pattern from the v0.2 playground: the `TemporaryDirectory`
object is held in a module-level variable so it isn't garbage-collected
before we're done with it.

In [1]:
import sqlite3
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import ar_db_handler as ardb

_TMP = tempfile.TemporaryDirectory(prefix="ar_db_paths_playground_")
WORKDIR = Path(_TMP.name)
FILINGS_PATH = WORKDIR / "filings.db"

print(f"ar_db_handler v{ardb.__version__}")
print(f"Working dir : {WORKDIR}")
print(f"filings.db  : {FILINGS_PATH}")


def now_iso() -> str:
    return datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")

ar_db_handler v0.4.0
Working dir : /tmp/ar_db_paths_playground__9ir7hcx
filings.db  : /tmp/ar_db_paths_playground__9ir7hcx/filings.db


## 1. Seed `filings.db`

We need two companies and a scraper run so that the auto-fill demos
later have something to look up against.

In [2]:
filings = ardb.init_filings_db(FILINGS_PATH)

ardb.upsert_company(
    filings,
    ardb.CompanyRecord(
        company_id=14778, fs_ticker="ACME", country_code="US",
        country="United States", country_id="US",
        file_name="acme", coverage_status="LAFA",
    ),
)
ardb.upsert_company(
    filings,
    ardb.CompanyRecord(
        company_id=200042, fs_ticker="TYO", country_code="JP",
        country="Japan", country_id="JP",
        file_name="tyo", coverage_status="LAFA",
    ),
)

scraper_id = ardb.make_run_id()
ardb.upsert_run(
    filings,
    ardb.RunRecord(
        scraper_id=scraper_id, country_code="US", workers_count=3,
        source_file="scripts/scrape_us.py", log_path=None, version="1.0.0",
        started_at=now_iso(), status="RUNNING",
    ),
)
print(f"Seeded 2 companies + 1 RUNNING scraper run (id={scraper_id[:8]}...)")

Seeded 2 companies + 1 RUNNING scraper run (id=8f57b202...)


## 2. The canonical path scheme

`ar-db-handler` owns one canonical GCS blob-path scheme. Every file
the pipeline writes ends up at:

```
rawdata/{country_code}/{company_id}/{fiscal_year}/{file_type}_{form_type}_{reporting_date}{extension}
```

`make_blob_path()` is the pure builder. It does **not** import from
`gcs-handler` — the path is just a string. `GCSClient` is bound to a
bucket at construction time, so the bucket name is not part of this
blob path.

`fiscal_year` is **trusted as stored on the record**. The H2/H1
derivation (covered in section 5) is the scrapers' responsibility at
record-building time; `make_blob_path()` only formats.

In [ ]:
# Three worked examples — these match the README and the test suite.
print(ardb.make_blob_path(
    country_code="US", company_id=14778, fiscal_year=2023,
    file_type="PDF", form_type="10-K", reporting_date="2023-12-31",
    extension=".pdf",
))
print(ardb.make_blob_path(
    country_code="US", company_id=14778, fiscal_year=2023,
    file_type="XBRL", form_type="10-K", reporting_date="2023-12-31",
    extension=".zip",
))
print(ardb.make_blob_path(
    country_code="JP", company_id=200042, fiscal_year=2023,
    file_type="PDF", form_type="ASR", reporting_date="2024-03-31",
    extension=".pdf",
))

rawdata/US/14778/2023/PDF_10-K_2023-12-31.pdf
rawdata/US/14778/2023/XBRL_10-K_2023-12-31.zip
rawdata/JP/200042/2023/PDF_ASR_2024-03-31.pdf


## 3. Validation — every component is checked

`make_blob_path` raises `ValueError` for each malformed input with a
message naming the offending field. `fiscal_year=None` is special: it
raises `MissingFiscalYearError`, the same exception `upsert_file()`
raises for the SUCCESS / `fiscal_year` invariant — one rule, one
exception type across the package.

A path with `UNKNOWN/` or `unknown-date` in it would pollute the
bucket layout and make deletion-by-prefix unsafe. That's why these
checks are strict. (The literal string `"UNKNOWN"` is allowed for
`form_type` only — it's a recognised value produced by
`_normalise_form_type`, not a missing one.)

In [4]:
def show_rejection(label, **bad_kwargs):
    base = dict(
        country_code="US", company_id=100, fiscal_year=2023,
        file_type="PDF", form_type="10-K", reporting_date="2023-12-31",
        extension=".pdf",
    )
    base.update(bad_kwargs)
    try:
        ardb.make_blob_path(**base)
    except (ValueError, ardb.MissingFiscalYearError) as e:
        cls = type(e).__name__
        print(f"  {label:30s}  → {cls}: {e}")


print("Each bad input triggers a specific, field-naming error:\n")
show_rejection("empty country_code", country_code="")
show_rejection("country_code with space", country_code="US X")
show_rejection("unknown file_type", file_type="HTML")
show_rejection("blank form_type", form_type="   ")
show_rejection("malformed reporting_date", reporting_date="31-12-2023")
show_rejection("extension without dot", extension="pdf")
show_rejection("fiscal_year=None", fiscal_year=None)

Each bad input triggers a specific, field-naming error:

  empty country_code              → ValueError: make_blob_path: country_code must be a non-empty string (got '')
  country_code with space         → ValueError: make_blob_path: country_code must not contain whitespace (got 'US X')
  unknown file_type               → ValueError: make_blob_path: file_type 'HTML' is not in EXTENSION_MAP (expected one of: PDF, XBRL)
  blank form_type                 → ValueError: make_blob_path: form_type must be a non-empty string (got '   ')
  malformed reporting_date        → ValueError: make_blob_path: reporting_date must match YYYY-MM-DD (got '31-12-2023')
  extension without dot           → ValueError: make_blob_path: extension must be non-empty and start with '.' (got 'pdf')
  fiscal_year=None                → MissingFiscalYearError: make_blob_path: fiscal_year must not be None — derive it from reporting_date before building the path.


## 4. `resolve_gcs_path()` — the `FileRecord` wrapper

`make_blob_path` takes seven raw arguments. In practice you usually
have a `FileRecord` in hand, so `resolve_gcs_path(record)` is the
convenience entry point. It:

* pulls `country_code`, `company_id`, `fiscal_year`, `file_type`,
  `form_type`, `reporting_date` off the record, AND
* resolves `extension` from `file_type` via `EXTENSION_MAP`
  (because `FileRecord` has no `extension` field — that lives on the
  DB row only).

Same exceptions as `make_blob_path`.

In [5]:
record = ardb.FileRecord(
    company_id=14778,
    scraper_id=scraper_id,
    status="SUCCESS",
    file_type="XBRL",
    source_filing_id="acc-demo",
    country_code="US",
    form_type="10-K",
    fiscal_year=2023,
    reporting_date="2023-12-31",
)
print(ardb.resolve_gcs_path(record))

rawdata/US/14778/2023/XBRL_10-K_2023-12-31.zip


## 5. `derive_fiscal_year()` — the H2/H1 rule

For scrapers building a `FileRecord`, `derive_fiscal_year` applies
the convention: a fiscal year is named by the calendar year that
contains its H2.

* period-end in months **07–12** → `fiscal_year = year(reporting_date)`
* period-end in months **01–06** → `fiscal_year = year(reporting_date) - 1`

`make_blob_path()` itself does **not** call this — it trusts the
value already on the record. One source of truth for the rule lives
in `paths.py`, callable from scrapers when they're building records.

In [6]:
samples = [
    ("2023-12-31", "Dec — H2"),
    ("2023-09-30", "Sep — H2"),
    ("2023-07-31", "Jul — H2 boundary"),
    ("2023-06-30", "Jun — H1 boundary"),
    ("2023-03-31", "Mar — H1"),
    ("2024-01-31", "Jan — H1 (rolls back)"),
    ("2024-03-31", "Mar — JP fiscal-year end"),
]
print(f"{'reporting_date':<14s}  {'note':<28s}  fiscal_year")
print("-" * 60)
for date, note in samples:
    fy = ardb.derive_fiscal_year(date)
    print(f"{date:<14s}  {note:<28s}  {fy}")

reporting_date  note                          fiscal_year
------------------------------------------------------------
2023-12-31      Dec — H2                      2023
2023-09-30      Sep — H2                      2023
2023-07-31      Jul — H2 boundary             2023
2023-06-30      Jun — H1 boundary             2022
2023-03-31      Mar — H1                      2022
2024-01-31      Jan — H1 (rolls back)         2023
2024-03-31      Mar — JP fiscal-year end      2023


## 6. The headline: `upsert_file()` auto-fills missing fields

This is what v0.4.0 actually buys you. On a SUCCESS row, when the
caller leaves `country_code` and/or `gcs_path` unset (or `None`),
`upsert_file()` resolves them automatically:

1. **`country_code`** is looked up from the `companies` table by
   `company_id`. (Cheap — single SELECT.)
2. **`gcs_path`** is built by calling `resolve_gcs_path(record)` on
   the now-complete record.

Both happen inside the same upsert, so the row written to `files`
has the resolved values.

The caller's `FileRecord` is **never mutated** — `upsert_file()` uses
`dataclasses.replace()` internally to build a new record before
writing. The caller's reference still has the original `None` values
after the call returns; that's a deliberate design choice for
referential transparency.

In [7]:
ardb.upsert_file(
    filings,
    ardb.FileRecord(
        company_id=14778,
        scraper_id=scraper_id,
        status="SUCCESS",
        file_type="PDF",
        source_filing_id="0000320193-24-acme-2023",
        form_type="10-K",
        fiscal_year=2023,
        reporting_date="2023-12-31",
        # country_code and gcs_path deliberately omitted.
    ),
)

row = filings.execute('''
    SELECT country_code, gcs_path, extension
    FROM files
    WHERE source_filing_id = '0000320193-24-acme-2023'
''').fetchone()
print(f"country_code (from companies): {row[0]!r}")
print(f"gcs_path     (canonical):      {row[1]!r}")
print(f"extension    (from file_type): {row[2]!r}")

country_code (from companies): 'US'
gcs_path     (canonical):      'rawdata/US/14778/2023/PDF_10-K_2023-12-31.pdf'
extension    (from file_type): '.pdf'


## 7. Caller-wins — explicit `gcs_path` is written verbatim

The auto-fill is a convenience, not a takeover. If you pass an
explicit `gcs_path`, it's written verbatim — useful for backfilling
files into a legacy bucket layout, or for any case where the scheme
doesn't apply.

In [8]:
ardb.upsert_file(
    filings,
    ardb.FileRecord(
        company_id=14778,
        scraper_id=scraper_id,
        status="SUCCESS",
        file_type="XBRL",
        source_filing_id="0000320193-24-acme-2023",
        form_type="10-K",
        fiscal_year=2023,
        reporting_date="2023-12-31",
        country_code="US",
        gcs_path="gs://legacy-bucket/imported/old-scheme/file.zip",  # ← explicit
    ),
)
row = filings.execute('''
    SELECT gcs_path FROM files
    WHERE source_filing_id = '0000320193-24-acme-2023'
      AND file_type = 'XBRL'
''').fetchone()
print(f"gcs_path (caller's, untouched): {row[0]!r}")

gcs_path (caller's, untouched): 'gs://legacy-bucket/imported/old-scheme/file.zip'


## 8. PENDING / FAILED rows are exempt

A `PENDING` row represents intent recorded before the metadata fetch
completed — it may legitimately lack `fiscal_year` and
`reporting_date` (the CHECK constraint allows NULL fiscal_year on
PENDING/FAILED). `gcs_path` is left at `NULL` in that case.

`country_code` is still auto-filled, because the `companies` lookup
is cheap and the row will need it when it eventually transitions to
`SUCCESS`.

In [9]:
ardb.upsert_file(
    filings,
    ardb.FileRecord(
        company_id=14778,
        scraper_id=scraper_id,
        status="PENDING",
        file_type="PDF",
        source_filing_id="0000320193-24-acme-2024",
        form_type=None,           # not yet known
        fiscal_year=None,         # legal on PENDING
        reporting_date=None,      # legal on PENDING
        country_code=None,        # still auto-filled — that's cheap
    ),
)
row = filings.execute('''
    SELECT status, country_code, fiscal_year, reporting_date, gcs_path
    FROM files
    WHERE source_filing_id = '0000320193-24-acme-2024'
''').fetchone()
print(f"status         : {row[0]}")
print(f"country_code   : {row[1]!r}   ← still resolved")
print(f"fiscal_year    : {row[2]}")
print(f"reporting_date : {row[3]}")
print(f"gcs_path       : {row[4]}    ← left NULL on PENDING")

status         : PENDING
country_code   : 'US'   ← still resolved
fiscal_year    : None
reporting_date : None
gcs_path       : None    ← left NULL on PENDING


## 9. Error paths — recorded to `scraper_errors` before re-raising

Same pattern as the rest of `ar-db-handler`: every rejection records
the failure to `scraper_errors` BEFORE raising. That keeps the audit
trail durable even when callers swallow exceptions.

Two new categories were added in v0.4.0:

* **`ERROR_FK_VIOLATION`** — `country_code` was `None` but
  `company_id` isn't in `companies`. We catch this earlier than
  SQLite's own FK message, with a clearer hint about the auto-fill
  origin.
* **`ERROR_MISSING_REPORTING_DATE`** — SUCCESS row, `gcs_path` was
  `None`, and `resolve_gcs_path()` couldn't build the path (usually
  because `reporting_date` is also `None`).

In [10]:
# Error path 1: SUCCESS + unknown company_id
try:
    ardb.upsert_file(
        filings,
        ardb.FileRecord(
            company_id=999_999,           # ← no such company
            scraper_id=scraper_id,
            status="SUCCESS",
            file_type="PDF",
            source_filing_id="acc-fk-demo",
            form_type="10-K",
            fiscal_year=2023,
            reporting_date="2023-12-31",
            country_code=None,            # forces the companies lookup
        ),
    )
except sqlite3.IntegrityError as e:
    print(f"caught IntegrityError:\n  {e}")

errs = ardb.get_scraper_errors(filings, error_type=ardb.ERROR_FK_VIOLATION)
print(f"\nFK_VIOLATION rows: {len(errs)}")
print(f"  company_id={errs[0]['company_id']}, "
      f"source_filing_id={errs[0]['source_filing_id']!r}")

caught IntegrityError:
  upsert_file: company_id=999999 not found in companies — cannot resolve country_code for source_filing_id='acc-fk-demo', file_type='PDF'. Upsert the company first.

FK_VIOLATION rows: 1
  company_id=999999, source_filing_id='acc-fk-demo'


In [11]:
# Error path 2: SUCCESS + reporting_date=None
try:
    ardb.upsert_file(
        filings,
        ardb.FileRecord(
            company_id=14778,
            scraper_id=scraper_id,
            status="SUCCESS",
            file_type="PDF",
            source_filing_id="acc-no-date",
            form_type="10-K",
            fiscal_year=2023,
            reporting_date=None,          # ← the offending field
            country_code=None,
        ),
    )
except ValueError as e:
    print(f"caught ValueError:\n  {e}")

errs = ardb.get_scraper_errors(filings, error_type=ardb.ERROR_MISSING_REPORTING_DATE)
print(f"\nMISSING_REPORTING_DATE rows: {len(errs)}")
print(f"  company_id={errs[0]['company_id']}, "
      f"source_filing_id={errs[0]['source_filing_id']!r}")

caught ValueError:
  make_blob_path: reporting_date must be a non-empty string (got '')

MISSING_REPORTING_DATE rows: 1
  company_id=14778, source_filing_id='acc-no-date'


## 10. Putting it together — the typical scraper flow

A realistic flow for a scraper after v0.4.0: fetch metadata from the
regulator, call `derive_fiscal_year`, build a partial `FileRecord`
(no `country_code`, no `gcs_path`), and let `upsert_file` handle the
rest.

This is the smallest possible call site for the common case. If you
need to override the scheme (e.g. backfilling), pass `country_code`
and/or `gcs_path` explicitly — caller wins, always.

In [12]:
def fetch_filing_metadata_from_edgar():
    """Stand-in for an EDGAR API call."""
    return {
        "accession": "0000320193-24-acme-q3",
        "period_end": "2024-09-30",
        "form_type": "10-Q",
    }


meta = fetch_filing_metadata_from_edgar()
fy = ardb.derive_fiscal_year(meta["period_end"])
print(f"reporting_date {meta['period_end']} → fiscal_year {fy}")

ardb.upsert_file(
    filings,
    ardb.FileRecord(
        company_id=14778,
        scraper_id=scraper_id,
        status="SUCCESS",
        file_type="PDF",
        source_filing_id=meta["accession"],
        form_type=meta["form_type"],
        fiscal_year=fy,
        reporting_date=meta["period_end"],
        # country_code + gcs_path auto-filled.
    ),
)

row = filings.execute(
    "SELECT country_code, gcs_path FROM files WHERE source_filing_id = ?",
    (meta["accession"],),
).fetchone()
print()
print("Row written to files:")
print(f"  country_code: {row[0]!r}")
print(f"  gcs_path:     {row[1]!r}")

reporting_date 2024-09-30 → fiscal_year 2024

Row written to files:
  country_code: 'US'
  gcs_path:     'rawdata/US/14778/2024/PDF_10-Q_2024-09-30.pdf'


## 11. Teardown

In [13]:
filings.close()
print("Done.")

Done.


---

### Cheatsheet — what `upsert_file()` does for you on SUCCESS rows

| Field           | Source                                            | Override                   |
|-----------------|---------------------------------------------------|----------------------------|
| `file_id`       | `make_file_id(company_id, source_filing_id, file_type)` | always derived; never set by caller |
| `extension`     | `EXTENSION_MAP[file_type]`                        | always derived; never set by caller |
| `form_type`     | `_normalise_form_type(record.form_type)` (blank → `"UNKNOWN"`) | always normalised |
| `country_code`  | `SELECT country_code FROM companies WHERE company_id=...` | caller wins if set        |
| `gcs_path`      | `resolve_gcs_path(record)`                        | caller wins if set        |

### Cheatsheet — new error categories in v0.4.0

| Trigger                                              | `error_type` constant            | Exception                |
|------------------------------------------------------|----------------------------------|--------------------------|
| `country_code=None`, `company_id` not in `companies` | `ERROR_FK_VIOLATION`             | `sqlite3.IntegrityError` |
| `gcs_path=None` on SUCCESS, missing path component   | `ERROR_MISSING_REPORTING_DATE`   | `ValueError`             |

### Cheatsheet — fiscal-year derivation (H2/H1)

| `reporting_date` month | `fiscal_year`               |
|------------------------|-----------------------------|
| Jul – Dec (07–12)      | `year(reporting_date)`      |
| Jan – Jun (01–06)      | `year(reporting_date) - 1`  |

Use `ar_db_handler.derive_fiscal_year(date)` in scrapers when
building `FileRecord`s. `make_blob_path()` does not call it — it
trusts the value on the record.